<a href="https://colab.research.google.com/github/jahnaviathi7/NewcodingBot/blob/main/Ai_health_assistant_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Symptom2Disease.csv to Symptom2Disease.csv
Saving Diseases_and_Symptoms_dataset.csv to Diseases_and_Symptoms_dataset.csv


In [ ]:
import pandas as pd

disease_df = pd.read_csv(
    "Diseases_and_Symptoms_dataset.csv"
)

print(disease_df.shape)

(96088, 231)


In [ ]:
X = disease_df.drop(
    "diseases",
    axis=1
)

y = disease_df["diseases"]

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_encoded = le.fit_transform(y)

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

RandomForestClassifier(n_jobs=-1, random_state=42)

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=50,
    max_depth=4,
    learning_rate=0.1,
    tree_method="hist",
    random_state=42
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=50,
              n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
print(X.isnull().sum().sum())

0


In [ ]:
print(X.dtypes.unique())

[dtype('int64')]


In [ ]:
X = X.apply(pd.to_numeric, errors='coerce')
X = X.fillna(0)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

print("Training Completed")

Training Completed


In [ ]:
from sklearn.metrics import accuracy_score

pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))

Accuracy: 0.8622125091060464


In [ ]:
import pandas as pd

nlp_df = pd.read_csv("Symptom2Disease.csv")

print(nlp_df.head())
print(nlp_df.shape)

   Unnamed: 0      label                                               text
0           0  Psoriasis  I have been experiencing a skin rash on my arm...
1           1  Psoriasis  My skin has been peeling, especially on my kne...
2           2  Psoriasis  I have been experiencing joint pain in my fing...
3           3  Psoriasis  There is a silver like dusting on my skin, esp...
4           4  Psoriasis  My nails have small dents or pits in them, and...
(1200, 3)


In [ ]:
import re

def clean_text(text):

    text = str(text).lower()

    text = re.sub(r'[^a-zA-Z\s]', '', text)

    return text

nlp_df["text"] = nlp_df["text"].apply(clean_text)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

X_text = vectorizer.fit_transform(
    nlp_df["text"]
)

y_text = nlp_df["label"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train_text,X_test_text,y_train_text,y_test_text = train_test_split(
    X_text,
    y_text,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.linear_model import LogisticRegression

nlp_model = LogisticRegression(
    max_iter=2000
)

nlp_model.fit(
    X_train_text,
    y_train_text
)

LogisticRegression(max_iter=2000)

In [ ]:
from sklearn.linear_model import LogisticRegression

nlp_model = LogisticRegression(
    max_iter=2000
)

nlp_model.fit(
    X_train_text,
    y_train_text
)

LogisticRegression(max_iter=2000)

In [ ]:
from sklearn.metrics import accuracy_score

pred_text = nlp_model.predict(
    X_test_text
)

acc_text = accuracy_score(
    y_test_text,
    pred_text
)

print("NLP Accuracy:",acc_text)

NLP Accuracy: 0.975


In [ ]:
import joblib

joblib.dump(
    nlp_model,
    "nlp_model.pkl"
)

joblib.dump(
    vectorizer,
    "tfidf_vectorizer.pkl"
)

['tfidf_vectorizer.pkl']

In [ ]:
def predict_from_text(user_text):

    user_text = clean_text(user_text)

    vec = vectorizer.transform(
        [user_text]
    )

    prediction = nlp_model.predict(
        vec
    )

    return prediction[0]

In [ ]:
predict_from_text(
    "I have fever headache and vomiting"
)

'Malaria'

In [ ]:
emergency_keywords = [

    "chest pain",

    "difficulty breathing",

    "shortness of breath",

    "heart attack",

    "stroke",

    "seizure",

    "unconscious",

    "bleeding"
]
def emergency_check(text):

    text = text.lower()

    for word in emergency_keywords:

        if word in text:

            return True

    return False

In [ ]:
def health_assistant():

    print("===== AI HEALTH ASSISTANT =====")

    while True:

        user = input("\nYou: ")

        if user.lower()=="exit":
            break

        if emergency_check(user):

            print(
                "\n⚠ Emergency detected."
            )

            print(
                "Seek immediate medical help."
            )

            continue

        disease = predict_from_text(user)

        print(
            "\nPredicted Disease:",
            disease
        )

In [ ]:
health_assistant()

===== AI HEALTH ASSISTANT =====

⚠ Emergency detected.
Seek immediate medical help.


In [ ]:
from sklearn.metrics import confusion_matrix

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 106.3 MB/s eta 0:00:00
